# Analiza danych F1 pod model decyzji o zmianie opon
**Zakres:** 2 tory — Austrian GP i Azerbaijan GP 2025, **tylko wyścigi** (Race), bo tylko tam są pit stopy i pełne stinty. **Źródło:** `TracingInsights-Archive/2025` (surowe JSON-y).

Cel analizy: znaleźć parametry powiązane ze **zużyciem opon** i **decyzją o pit stopie**, żeby potem wybrać cechy do modelu, który podpowie *kiedy zmienić opony*.

**Cache (dane pobierane tylko raz):**
1. Przy pierwszym uruchomieniu RAW jest klonowany do tymczasowego `/content` (nie na Drive), parsowany i zapisywany jako parquet w `MyDrive/F1-Research/cache/`.
2. Każde kolejne uruchomienie tylko wczytuje parquet — bez pobierania i bez parsowania (kilka sekund).
3. Nowa runda w `EVENTS` → pobierana jest tylko ta brakująca. `FORCE_DOWNLOAD=True` wymusza pełne odświeżenie.

Wyniki (CSV + PNG) lądują w `MyDrive/F1-Research/reports/`, a na końcu notebook pakuje je do ZIP i pobiera na komputer.

In [ ]:
# Colab ma już pandas/numpy/scipy/seaborn/statsmodels/sklearn; doinstalowujemy tylko brakujące
import importlib.util, subprocess, sys
need=[p for p,m in [('pyarrow','pyarrow'),('statsmodels','statsmodels'),('scikit-learn','sklearn'),('seaborn','seaborn')] if importlib.util.find_spec(m) is None]
if need: subprocess.run([sys.executable,'-m','pip','-q','install',*need],check=True)
print('Doinstalowano:',need or 'nic')

## 1. Konfiguracja i ścieżki

In [ ]:
from pathlib import Path
import os, shutil, json, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 200)

YEAR=2025
EVENTS=['Austrian Grand Prix','Azerbaijan Grand Prix']
REPO='https://github.com/TracingInsights-Archive/2025.git'

FORCE_DOWNLOAD=False          # True = pobierz i przetwórz RAW od nowa mimo istniejącego cache
KEEP_RAW=False                # True = zachowaj też surowe JSON-y na Drive (~280 MB, kopiowanie na Drive jest wolne)
SAVE_TELEMETRY_SAMPLES=True   # pełne próbki telemetrii (wszystkie okrążenia) w parquet - pod przyszłe modele
DOWNLOAD_REPORTS_ZIP=True     # na końcu pobierz ZIP z raportami na komputer

try:
    from google.colab import drive, files as colab_files
    IN_COLAB=True
except ImportError:
    IN_COLAB=False

if IN_COLAB:
    drive.mount('/content/drive')
    ROOT=Path('/content/drive/MyDrive/F1-Research')
    TMP=Path('/content/f1-source')          # dysk lokalny maszyny Colab - szybki, znika po sesji
else:
    ROOT=Path(os.environ.get('F1_ROOT', Path.cwd()/'F1-Research'))
    TMP=ROOT/'_tmp_source'
CACHE=ROOT/'cache'/str(YEAR)
RAW=ROOT/'raw'/str(YEAR)
REPORTS=ROOT/'reports'
for p in (CACHE, REPORTS): p.mkdir(parents=True, exist_ok=True)
print('Colab:',IN_COLAB,'| folder projektu:',ROOT)

# Kolory: mieszanki i tory (zwalidowane pod daltonizm; kolor zawsze z podpisem w legendzie)
PAL={'SOFT':'#e34948','MEDIUM':'#eda100','HARD':'#2a78d6'}
EV_PAL={'Austrian Grand Prix':'#2a78d6','Azerbaijan Grand Prix':'#eb6834'}
COMP_ORDER=['SOFT','MEDIUM','HARD']
sns.set_theme(style='whitegrid',context='notebook',rc={'axes.edgecolor':'#bbbbbb','grid.color':'#e6e6e6'})

## 2. Pobranie RAW (tylko gdy brak cache) → parquet
Z każdej rundy powstają 4 pliki:
- `*_laps.parquet` — jedno okrążenie = jeden wiersz (czasy, opony, pit, pozycja, status toru, pogoda),
- `*_lapfeat.parquet` — cechy telemetrii policzone dla każdego okrążenia (gaz, hamulec, przeciążenia, prędkości w zakrętach, jazda za rywalem),
- `*_corners.parquet` — minimalna prędkość w każdym zakręcie na każdym okrążeniu,
- `*_samples.parquet` — pełne próbki telemetrii (opcjonalnie).

`PARSER_VERSION` zwiększamy, gdy zmieni się sposób liczenia cech — wtedy cache przebuduje się sam.

In [ ]:
PARSER_VERSION=2   # v2: DRS w tych danych to 0/1 (nie kody FastF1 10/12/14)
CORNER_WINDOW_M=60    # okno ±m wokół wierzchołka zakrętu przy szukaniu prędkości minimalnej
DIRTY_AIR_S=1.0       # odstęp [s] do auta z przodu uznawany za jazdę w brudnym powietrzu

LAP_COLS={'drv':'Driver','dNum':'DriverNumber','team':'Team','lap':'LapNumber','time':'LapTime',
          's1':'Sector1Time','s2':'Sector2Time','s3':'Sector3Time','compound':'Compound','stint':'Stint',
          'life':'TyreLife','fresh':'FreshTyre','pos':'Position','status':'TrackStatus',
          'pout':'PitOutTime','pin':'PitInTime','lST':'LapStartSessionTime','sesT':'LapEndSessionTime',
          'vi1':'SpeedI1','vi2':'SpeedI2','vfl':'SpeedFL','vst':'SpeedST','del':'Deleted','iacc':'IsAccurate',
          'pb':'IsPersonalBest','wAT':'AirTemp','wTT':'TrackTemp','wH':'Humidity','wP':'Pressure',
          'wR':'Rainfall','wWS':'WindSpeed','wWD':'WindDirection'}
TEXT_COLS={'Driver','DriverNumber','Team','Compound','TrackStatus'}
BOOL_COLS={'FreshTyre','Deleted','IsAccurate','IsPersonalBest','Rainfall'}

def slug(ev): return ev.split()[0].lower()
def cache_files(ev):
    s=slug(ev)
    return {k:CACHE/f'{s}_{k}.parquet' for k in ('laps','lapfeat','corners','samples')} | {'meta':CACHE/f'{s}_meta.json'}

def is_cached(ev):
    f=cache_files(ev)
    if not f['meta'].exists(): return False
    meta=json.loads(f['meta'].read_text(encoding='utf8'))
    needed=['laps','lapfeat','corners']+(['samples'] if SAVE_TELEMETRY_SAMPLES else [])
    return meta.get('parser_version')==PARSER_VERSION and all(f[k].exists() for k in needed)

def none_to_nan(values):
    return [np.nan if v is None or v=='None' else v for v in values]

def read_json(path): return json.loads(Path(path).read_text(encoding='utf8'))

def fetch_raw(events):
    """Zwraca {event: folder Race}. Używa kopii RAW z Drive, jeśli jest; brakujące klonuje raz (sparse)."""
    out={}
    todo=[ev for ev in events if FORCE_DOWNLOAD or not (RAW/ev/'Race'/'drivers.json').exists()]
    for ev in events:
        if ev not in todo: out[ev]=RAW/ev/'Race'
    if todo:
        if TMP.exists(): shutil.rmtree(TMP)
        git=['git','-c','core.longpaths=true']
        subprocess.run([*git,'clone','-q','--depth','1','--filter=blob:none','--sparse',REPO,str(TMP)],check=True)
        subprocess.run([*git,'-C',str(TMP),'sparse-checkout','set','--',*[f'{ev}/Race' for ev in todo]],check=True)
        commit=subprocess.check_output(['git','-C',str(TMP),'rev-parse','HEAD'],text=True).strip()
        print('Pobrano RAW, commit:',commit)
        for ev in todo:
            src=TMP/ev/'Race'
            if not src.is_dir(): raise RuntimeError(f'Brak {src} - sprawdź nazwę rundy w repozytorium.')
            if KEEP_RAW:
                dst=RAW/ev/'Race'
                if dst.exists(): shutil.rmtree(dst)
                shutil.copytree(src,dst); out[ev]=dst
            else:
                out[ev]=src
        fetch_raw.commit=commit
    return out

def parse_laps(race_dir, ev):
    frames=[]
    for f in sorted(race_dir.glob('*/laptimes.json')):
        x=read_json(f)
        frames.append(pd.DataFrame({new:none_to_nan(x[old]) for old,new in LAP_COLS.items() if old in x}))
    laps=pd.concat(frames,ignore_index=True)
    for c in laps.columns:
        if c in TEXT_COLS: laps[c]=laps[c].astype('string')
        elif c in BOOL_COLS: laps[c]=laps[c].astype('boolean')
        else: laps[c]=pd.to_numeric(laps[c],errors='coerce')
    laps.insert(0,'Event',ev)
    return laps.sort_values(['Driver','LapNumber']).reset_index(drop=True)

def lap_telemetry_features(t, corner_dist):
    """Agregaty jednego okrążenia. Udziały liczone wagą czasu (próbki nie są równomierne)."""
    t=t.sort_values('time')
    tm=t['time'].to_numpy(float); dt=np.diff(tm,prepend=tm[0]); total=dt.sum()
    if total<=0 or len(t)<20: return None, None
    share=lambda mask: float(dt[mask].sum()/total)
    v=t['speed'].to_numpy(float); thr=t['throttle'].to_numpy(float); brk=t['brake'].to_numpy(float)>0
    gear=t['gear'].to_numpy(float); drs=t['drs'].to_numpy(float); rpm=t['rpm'].to_numpy(float)
    ax=t['acc_x'].to_numpy(float); ay=t['acc_y'].to_numpy(float); dist=t['distance'].to_numpy(float)
    vms=v/3.6
    f={'Tel_Duration':total,
       'Speed_mean':float(np.nansum(v*dt)/total),'Speed_max':np.nanmax(v),'Speed_p10':np.nanpercentile(v,10),
       'Throttle_mean':float(np.nansum(thr*dt)/total),'FullThrottle_share':share(thr>=98),
       'Brake_share':share(brk),'Brake_events':int(np.sum(brk[1:]&~brk[:-1])),
       'Coast_share':share((thr<5)&~brk),
       'GearShifts':int(np.sum(np.diff(gear)!=0)),'RPM_mean':float(np.nansum(rpm*dt)/total),
       'DRS_share':share((drs==1)|(drs>=10)),
       'LatAcc_absmean':float(np.nansum(np.abs(ay)*dt)/total),'LatAcc_p95':np.nanpercentile(np.abs(ay),95),
       'BrakeDecel_p05':np.nanpercentile(ax,5),'TractionAcc_p95':np.nanpercentile(ax,95),
       # "energia" przekazywana przez opony: |a| * droga (proxy obciążenia opon)
       'LatLoad':float(np.nansum(np.abs(ay)*vms*dt)/1000),'LongLoad':float(np.nansum(np.abs(ax)*vms*dt)/1000)}
    ahead=t['DriverAhead'].notna().to_numpy() & (t['DriverAhead'].astype(str)!='None').to_numpy()
    gap_s=t['DistanceToDriverAhead'].to_numpy(float)/np.maximum(vms,1)
    f['DirtyAir_share']=share(ahead&(gap_s<DIRTY_AIR_S))
    f['GapAhead_median_s']=float(np.nanmedian(np.where(ahead,gap_s,np.nan))) if ahead.any() else np.nan
    corners=[]
    for d in corner_dist:
        m=np.abs(dist-d)<=CORNER_WINDOW_M
        corners.append(np.nanmin(v[m]) if m.any() else np.nan)
    f['CornerMinSpeed_mean']=float(np.nanmean(corners)) if np.isfinite(corners).any() else np.nan
    return f, corners

def parse_telemetry(race_dir, ev):
    cr=read_json(race_dir/'corners.json')
    corner_num=cr['CornerNumber']; corner_dist=cr['Distance']
    feats=[]; crn=[]; samples=[]
    for drv_dir in sorted(p for p in race_dir.iterdir() if p.is_dir()):
        for f in drv_dir.glob('*_tel.json'):
            lap=int(f.name.split('_')[0])
            t=pd.DataFrame(read_json(f)['tel']).drop(columns=['dataKey'],errors='ignore')
            t['DriverAhead']=t['DriverAhead'].replace('None',np.nan)
            for c in t.columns:
                if c!='DriverAhead': t[c]=pd.to_numeric(t[c].replace('None',np.nan),errors='coerce')
            fe,cm=lap_telemetry_features(t,corner_dist)
            if fe is None: continue
            key={'Event':ev,'Driver':drv_dir.name,'LapNumber':lap}
            feats.append(key|fe)
            crn.append(key|{f'T{n:02d}':s for n,s in zip(corner_num,cm)})
            if SAVE_TELEMETRY_SAMPLES:
                t.insert(0,'LapNumber',lap); t.insert(0,'Driver',drv_dir.name); t.insert(0,'Event',ev)
                samples.append(t)
    smp=pd.concat(samples,ignore_index=True) if samples else None
    return pd.DataFrame(feats), pd.DataFrame(crn), smp

def build_event(ev, race_dir):
    f=cache_files(ev)
    laps=parse_laps(race_dir,ev)
    feat,crn,smp=parse_telemetry(race_dir,ev)
    laps.to_parquet(f['laps'],index=False); feat.to_parquet(f['lapfeat'],index=False); crn.to_parquet(f['corners'],index=False)
    if smp is not None:
        smp['DriverAhead']=smp['DriverAhead'].astype('string')
        smp.to_parquet(f['samples'],index=False,compression='zstd')
    meta={'event':ev,'year':YEAR,'parser_version':PARSER_VERSION,'repository':REPO,
          'commit':getattr(fetch_raw,'commit',None),'n_laps':len(laps),'n_tel_laps':len(feat)}
    f['meta'].write_text(json.dumps(meta,indent=2),encoding='utf8')
    print(f'{ev}: {len(laps)} okrążeń, telemetria dla {len(feat)} okrążeń -> zapisano cache')

missing=[ev for ev in EVENTS if FORCE_DOWNLOAD or not is_cached(ev)]
if missing:
    print('Brak cache dla:',missing,'-> pobieram RAW')
    sources=fetch_raw(missing)
    for ev in missing: build_event(ev,sources[ev])
    if TMP.exists() and not KEEP_RAW: shutil.rmtree(TMP,ignore_errors=True)
else:
    print('Cache kompletny - nic nie pobieram.')

laps=pd.concat([pd.read_parquet(cache_files(ev)['laps']) for ev in EVENTS],ignore_index=True)
lapfeat=pd.concat([pd.read_parquet(cache_files(ev)['lapfeat']) for ev in EVENTS],ignore_index=True)
corners={ev:pd.read_parquet(cache_files(ev)['corners']) for ev in EVENTS}
for ev in EVENTS: print(json.loads(cache_files(ev)['meta'].read_text(encoding='utf8')))
print({p.name:f'{p.stat().st_size/1024**2:.1f} MB' for p in sorted(CACHE.glob('*.parquet'))})

## 3. Jeden zbiór: okrążenia + cechy telemetrii
**Czyste okrążenie** (do badania tempa i zużycia opon): zielona flaga (`TrackStatus=='1'`), nie 1. okrążenie, nie in-lap/out-lap, nieusunięte, czas < 107% mediany czystych okrążeń rundy.
Okrążeń nieczystych **nie wyrzucamy** — są potrzebne do analizy decyzji o pit stopie (np. zjazd pod Safety Car).

In [ ]:
df=laps.merge(lapfeat,on=['Event','Driver','LapNumber'],how='left',validate='one_to_one')
df['IsPitInLap']=df['PitInTime'].notna()
df['IsPitOutLap']=df['PitOutTime'].notna()
df['IsGreen']=df['TrackStatus'].eq('1').fillna(False)
df['UnderSC_VSC']=df['TrackStatus'].fillna('').str.contains('[4567]',regex=True)   # 4=SC, 5=czerwona, 6/7=VSC
base=df['IsGreen']&~df['IsPitInLap']&~df['IsPitOutLap']&(df['LapNumber']>1)&~df['Deleted'].fillna(False)&df['LapTime'].notna()
med=df[base].groupby('Event')['LapTime'].median()
df['Clean']=base&(df['LapTime']<1.07*df['Event'].map(med))
# etykieta pod model: czy kierowca zjedzie do boksu na koniec NASTĘPNEGO okrążenia
df=df.sort_values(['Event','Driver','LapNumber']).reset_index(drop=True)
df['PitNextLap']=df.groupby(['Event','Driver'])['IsPitInLap'].shift(-1).fillna(False).astype(bool)
df['RaceProgress']=df['LapNumber']/df.groupby('Event')['LapNumber'].transform('max')

overview=df.groupby('Event').agg(laps=('LapNumber','size'),drivers=('Driver','nunique'),race_laps=('LapNumber','max'),
    clean_laps=('Clean','sum'),pit_stops=('IsPitInLap','sum'),laps_SC_VSC=('UnderSC_VSC','sum'),
    telemetry_coverage=('Speed_mean',lambda s:round(s.notna().mean(),3)))
display(overview); overview.to_csv(REPORTS/'overview.csv')
display(df.groupby(['Event','Compound']).size().unstack(fill_value=0))

## 4. Rozdzielenie efektu paliwa i zużycia opon
W trakcie stintu `TyreLife` i `LapNumber` rosną razem — auto przyspiesza, bo spala paliwo, a zwalnia, bo zużywa opony. Te dwa efekty da się rozdzielić tylko dlatego, że pit stop zeruje `TyreLife`, a `LapNumber` rośnie dalej.

Model OLS na czystych okrążeniach (osobno dla każdej rundy):
`LapTime ~ LapNumber + C(Compound) + C(Compound):TyreLife + C(Driver)`
- współczynnik `LapNumber` = efekt paliwa [s/okr.],
- `Compound:TyreLife` = **degradacja danej mieszanki** [s/okr.] — kluczowa wielkość dla strategii,
- `C(Driver)` usuwa różnice tempa kierowców i bolidów.

In [ ]:
import statsmodels.formula.api as smf
deg_rows=[]; FUEL={}
for ev in EVENTS:
    d=df[df.Event.eq(ev)&df.Clean].copy()
    d['Compound']=d['Compound'].astype(str); d['Driver']=d['Driver'].astype(str)
    m=smf.ols('LapTime ~ LapNumber + C(Compound) + C(Compound):TyreLife + C(Driver)',data=d).fit()
    FUEL[ev]=m.params['LapNumber']
    deg_rows.append({'Event':ev,'term':'paliwo (LapNumber)','coef_s_per_lap':m.params['LapNumber'],'p':m.pvalues['LapNumber'],'n_laps':len(d)})
    for k in m.params.index:
        if k.endswith(':TyreLife'):
            deg_rows.append({'Event':ev,'term':'degradacja '+k.split('[')[1].split(']')[0],'coef_s_per_lap':m.params[k],'p':m.pvalues[k],
                             'n_laps':int((d.Compound==k.split('[')[1].split(']')[0]).sum())})
    print(f'{ev}: R²={m.rsquared:.3f}')
deg=pd.DataFrame(deg_rows); display(deg.round(4)); deg.to_csv(REPORTS/'fuel_and_degradation_model.csv',index=False)

# Czas skorygowany o paliwo i strata względem początku stintu (= zmienna "zużycie opon")
df['LapTimeFuelCorr']=df['LapTime']-df['Event'].map(FUEL)*(df['LapNumber']-1)
df['LapTimeNorm']=df['LapTime']/df['Event'].map(med)
def stint_baseline(g):
    c=g[g.Clean].sort_values('LapNumber')['LapTimeFuelCorr']
    return c.iloc[:3].median() if len(c)>=2 else np.nan
bl=df.groupby(['Event','Driver','Stint']).apply(stint_baseline,include_groups=False).rename('StintBaseline')
df=df.merge(bl.reset_index(),on=['Event','Driver','Stint'],how='left')
df['DegDelta']=df['LapTimeFuelCorr']-df['StintBaseline']     # [s] ile wolniej niż na początku stintu (po korekcie paliwa)

## 5. Krzywe degradacji i przebieg stintów

In [ ]:
fig,axes=plt.subplots(1,len(EVENTS),figsize=(7*len(EVENTS),4.5),sharey=True)
for ax,ev in zip(np.atleast_1d(axes),EVENTS):
    d=df[df.Event.eq(ev)&df.Clean&df.DegDelta.notna()]
    for comp,g in d.groupby('Compound'):
        ax.scatter(g.TyreLife,g.DegDelta,s=7,alpha=.25,color=PAL.get(comp,'k'))
        mean=g.groupby('TyreLife').DegDelta.agg(['median','size']); mean=mean[mean['size']>=5]
        ax.plot(mean.index,mean['median'],color=PAL.get(comp,'k'),lw=2.5,label=f'{comp} (mediana)')
    ax.axhline(0,color='k',lw=.6); ax.set_ylim(-2,4)
    ax.set(title=f'{ev}: strata vs wiek opony',xlabel='TyreLife [okr.]',ylabel='DegDelta [s] (po korekcie paliwa)'); ax.legend()
plt.tight_layout(); plt.savefig(REPORTS/'degradation_curves.png',dpi=160); plt.show()

# Tabela stintów: długość, nachylenie degradacji, czy zjazd pod SC
def stint_summary(g):
    c=g[g.Clean&g.DegDelta.notna()]
    slope=np.polyfit(c.TyreLife,c.DegDelta,1)[0] if len(c)>=5 else np.nan
    last=g.sort_values('LapNumber').iloc[-1]
    return pd.Series({'Compound':g.Compound.iloc[0],'start_lap':g.LapNumber.min(),'end_lap':g.LapNumber.max(),
        'length':len(g),'start_tyre_life':g.TyreLife.min(),'clean_laps':len(c),'deg_slope_s_per_lap':slope,
        'ended_by_pit':bool(last.IsPitInLap),'pit_under_SC_VSC':bool(last.IsPitInLap and last.UnderSC_VSC),
        'DegDelta_last3':c.sort_values('LapNumber').DegDelta.tail(3).mean()})
stints=df.groupby(['Event','Driver','Stint']).apply(stint_summary,include_groups=False).reset_index()
stints.to_csv(REPORTS/'stints.csv',index=False)
display(stints.groupby(['Event','Compound']).agg(n_stints=('length','size'),median_length=('length','median'),
    max_length=('length','max'),median_slope=('deg_slope_s_per_lap','median'),pits_under_SC=('pit_under_SC_VSC','sum')).round(3))

## 6. Kandydaci na cechy i parametry nadmiarowe
Zestaw bez duplikatów i bez tautologii: nie bierzemy osobno czasów sektorów do celu `LapTime` (suma), ani kilku zegarów sesji (to ta sama zmienna co `LapNumber`).
- **Nadmiarowe pary** (|ρ| ≥ 0.85) — do modelu wystarczy jedna z pary.
- **Clustermap** pokazuje grupy parametrów niosących tę samą informację.

In [ ]:
CONTEXT=['LapNumber','RaceProgress','TyreLife','Stint','Position','AirTemp','TrackTemp','Humidity','Pressure','WindSpeed']
SPEED_TRAPS=['SpeedI1','SpeedI2','SpeedFL','SpeedST']
TELEMETRY=[c for c in lapfeat.columns if c not in ('Event','Driver','LapNumber','Tel_Duration')]
TARGETS=['LapTime','LapTimeFuelCorr','LapTimeNorm','DegDelta']
FEATURES=CONTEXT+SPEED_TRAPS+TELEMETRY
GROUP={**{c:'kontekst' for c in CONTEXT},**{c:'pułapki prędkości' for c in SPEED_TRAPS},**{c:'telemetria' for c in TELEMETRY}}

red_rows=[]
for ev in EVENTS:
    x=df[df.Event.eq(ev)&df.Clean][FEATURES+TARGETS]
    x=x.loc[:,x.nunique()>1]
    rho=x.corr(method='spearman',min_periods=30)
    rho.to_csv(REPORTS/f'{slug(ev)}_spearman_clean.csv')
    cg=sns.clustermap(rho.fillna(0),cmap='vlag',vmin=-1,vmax=1,figsize=(13,13),xticklabels=True,yticklabels=True)
    cg.fig.suptitle(f'{ev}: Spearman, czyste okrążenia (grupy = parametry niosące tę samą informację)',y=1.01)
    cg.savefig(REPORTS/f'{slug(ev)}_clustermap.png',dpi=140); plt.show()
    cols=[c for c in rho.columns if c in FEATURES]
    for i,a in enumerate(cols):
        for b in cols[i+1:]:
            r=rho.loc[a,b]
            if pd.notna(r) and abs(r)>=0.85: red_rows.append({'Event':ev,'param_a':a,'param_b':b,'rho':round(r,3)})
redundant=pd.DataFrame(red_rows)
redundant_wide=redundant.pivot_table(index=['param_a','param_b'],columns='Event',values='rho').reset_index()
redundant_wide.to_csv(REPORTS/'redundant_pairs.csv',index=False)
display(redundant_wide)

## 7. Co mówi telemetria o zużyciu opon?
Dla każdej cechy liczymy dwa wskaźniki:
1. **ρ w stincie** — mediana korelacji Spearmana cechy z `TyreLife` liczona osobno w każdym stincie (≥8 czystych okrążeń), plus odsetek stintów z tym samym znakiem. Uwaga: w stincie wiek opony = postęp wyścigu, więc to mieszanka zużycia i spalania paliwa.
2. **t(TyreLife) z regresji** `cecha ~ TyreLife + LapNumber + C(Driver)` — efekt wieku opony **po odjęciu efektu paliwa i kierowcy** (standaryzowany, porównywalny między cechami).

Cecha, która ma duże |t| w obu rundach i ten sam znak, jest dobrym kandydatem na wskaźnik zużycia opon w modelu.

**Pułapki interpretacji:** `Position` „poprawia się” z wiekiem opony, bo rywale zjeżdżają do boksu (cykl pit stopów) — to efekt strategii, nie zużycia. `TrackTemp` spada w trakcie wyścigu, więc jej związek z `TyreLife` to głównie upływ czasu. Dlatego za wskaźniki zużycia uznajemy tylko sygnały z samochodu (telemetria, pułapki prędkości).

In [ ]:
def within_stint_rho(d, feat, min_laps=8):
    vals=[]
    for _,g in d.groupby(['Driver','Stint']):
        g=g[[feat,'TyreLife']].dropna()
        if len(g)>=min_laps and g[feat].nunique()>1: vals.append(g[feat].corr(g['TyreLife'],method='spearman'))
    vals=pd.Series(vals,dtype=float)
    if vals.empty: return np.nan,np.nan,0
    med_=vals.median()
    return med_, (np.sign(vals)==np.sign(med_)).mean(), len(vals)

wear_rows=[]
for ev in EVENTS:
    d=df[df.Event.eq(ev)&df.Clean].copy(); d['Driver']=d['Driver'].astype(str)
    for feat in FEATURES+['LapTimeFuelCorr']:
        if feat in ('TyreLife','LapNumber','RaceProgress','Stint') or d[feat].nunique()<=2: continue
        med_,cons,n=within_stint_rho(d,feat)
        z=d[[feat,'TyreLife','LapNumber','Driver']].dropna()
        z=z.assign(y=(z[feat]-z[feat].mean())/z[feat].std())
        try:
            m=smf.ols('y ~ TyreLife + LapNumber + C(Driver)',data=z).fit()
            t,beta,p=m.tvalues['TyreLife'],m.params['TyreLife'],m.pvalues['TyreLife']
        except Exception: t=beta=p=np.nan
        wear_rows.append({'Event':ev,'feature':feat,'group':GROUP.get(feat,'cel'),'rho_in_stint':med_,'sign_consistency':cons,
                          'n_stints':n,'beta_std_per_lap':beta,'t_TyreLife':t,'p_TyreLife':p})
wear=pd.DataFrame(wear_rows)
wear_wide=wear.pivot_table(index=['feature','group'],columns='Event',values=['t_TyreLife','rho_in_stint']).round(3)
wear_wide.columns=[f'{a}|{slug(b)}' for a,b in wear_wide.columns]
tcols=[c for c in wear_wide if c.startswith('t_')]
wear_wide['same_sign_both']=np.sign(wear_wide[tcols]).nunique(axis=1).eq(1)
wear_wide['min_abs_t']=wear_wide[tcols].abs().min(axis=1)
wear_wide=wear_wide.sort_values(['same_sign_both','min_abs_t'],ascending=False).reset_index()
wear_wide.to_csv(REPORTS/'tyre_wear_indicators.csv',index=False)
display(wear_wide.head(25))

top=wear_wide[wear_wide.same_sign_both].head(15)['feature']
plot=wear[wear.feature.isin(top)]
plt.figure(figsize=(10,6))
sns.barplot(data=plot,y='feature',x='t_TyreLife',hue='Event',order=list(top))
plt.axvline(0,color='k',lw=.6); plt.axvline(2,ls=':',color='grey'); plt.axvline(-2,ls=':',color='grey')
plt.title('Wpływ wieku opony na cechę (t-stat, kontrola paliwa i kierowcy)')
plt.tight_layout(); plt.savefig(REPORTS/'tyre_wear_indicators.png',dpi=160); plt.show()

### 7a. Prędkość w zakrętach a wiek opony
Zużyta opona traci przyczepność, co powinno być widać w **prędkości minimalnej w zakrętach**, a mniej na prostych. Zakręty dzielimy na wolne/średnie/szybkie według mediany prędkości minimalnej w danej rundzie.

In [ ]:
corner_rows=[]
for ev in EVENTS:
    c=corners[ev].merge(df[['Event','Driver','LapNumber','Clean','TyreLife','Driver']].loc[:,lambda x:~x.columns.duplicated()],
                        on=['Event','Driver','LapNumber'])
    c=c[c.Clean]
    tc=[k for k in c.columns if k.startswith('T') and k[1:].isdigit()]
    for k in tc:
        z=c[[k,'TyreLife','LapNumber','Driver']].dropna(); z['Driver']=z['Driver'].astype(str)
        if len(z)<50: continue
        m=smf.ols(f'{k} ~ TyreLife + LapNumber + C(Driver)',data=z).fit()
        corner_rows.append({'Event':ev,'corner':k,'median_min_speed':z[k].median(),
                            'kmh_per_tyre_lap':m.params['TyreLife'],'t_TyreLife':m.tvalues['TyreLife']})
corner_eff=pd.DataFrame(corner_rows)
corner_eff['type']=pd.cut(corner_eff.median_min_speed,[0,130,200,400],labels=['wolny <130','średni 130-200','szybki >200'])
corner_eff.to_csv(REPORTS/'corner_speed_vs_tyre_life.csv',index=False)
display(corner_eff.groupby(['Event','type'],observed=True)[['kmh_per_tyre_lap','t_TyreLife']].median().round(3))
plt.figure(figsize=(9,4.5))
sns.scatterplot(data=corner_eff,x='median_min_speed',y='kmh_per_tyre_lap',hue='Event',s=60)
for _,r in corner_eff.iterrows(): plt.annotate(r.corner,(r.median_min_speed,r.kmh_per_tyre_lap),fontsize=7,alpha=.7)
plt.axhline(0,color='k',lw=.6); plt.title('Zmiana prędkości min. w zakręcie na okrążenie wieku opony (kontrola paliwa i kierowcy)')
plt.xlabel('mediana prędkości minimalnej [km/h]'); plt.ylabel('km/h na okr. wieku opony')
plt.tight_layout(); plt.savefig(REPORTS/'corner_speed_vs_tyre_life.png',dpi=160); plt.show()

## 7b. Wykresy powiązań parametrów (2 tory, tylko wyścigi)
Kluczowe parametry dla decyzji o oponach:
- **zużycie:** `TyreLife`, `DegDelta`, `LapTimeFuelCorr`,
- **sygnały z auta:** `CornerMinSpeed_mean`, `LatAcc_absmean`, `Speed_mean`, `SpeedST`, `Throttle_mean`,
- **warunki:** `TrackTemp`, `DirtyAir_share`.

1. Heatmapa Spearmana (te same parametry, oba tory obok siebie + różnica) — gdzie zależność jest wspólna, a gdzie zależy od toru.
2. Parametr vs wiek opony, osobno dla mieszanek — jak wygląda zależność, a nie tylko jej siła.
3. Macierz rozrzutu (pairplot) dla każdego toru.
4. Zgodność torów: efekt wieku opony w Austrii vs w Baku.

In [ ]:
KEY=['TyreLife','DegDelta','LapTimeFuelCorr','CornerMinSpeed_mean','LatAcc_absmean','Speed_mean','SpeedST',
     'Throttle_mean','TrackTemp','DirtyAir_share','LapNumber']
rho_key={ev:df[df.Event.eq(ev)&df.Clean][KEY].corr(method='spearman') for ev in EVENTS}
fig,axes=plt.subplots(1,3,figsize=(22,7))
for ax,(title,m) in zip(axes,[(EVENTS[0],rho_key[EVENTS[0]]),(EVENTS[1],rho_key[EVENTS[1]]),
                              ('Różnica (Austria − Baku)',rho_key[EVENTS[0]]-rho_key[EVENTS[1]])]):
    mask=np.triu(np.ones_like(m,dtype=bool))
    sns.heatmap(m,mask=mask,ax=ax,cmap='vlag',center=0,vmin=-1,vmax=1,annot=True,fmt='.2f',annot_kws={'size':8},
                linewidths=2,linecolor='white',cbar_kws={'shrink':.7,'label':'Δρ' if 'Różnica' in title else 'ρ Spearmana'})
    ax.set_title(title)
plt.suptitle('Powiązania kluczowych parametrów — czyste okrążenia wyścigu',y=1.02,fontsize=14)
plt.tight_layout(); plt.savefig(REPORTS/'key_params_heatmaps.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
# Parametr vs wiek opony: punkty = okrążenia, linia = mediana dla danego wieku opony (min. 5 okrążeń)
REL=[('DegDelta','strata vs start stintu [s]'),('CornerMinSpeed_mean','śr. prędkość min. w zakrętach [km/h]'),
     ('LatAcc_absmean','śr. |przyspieszenie boczne| [m/s²]'),('Speed_mean','średnia prędkość [km/h]'),('TrackTemp','temp. toru [°C]')]
fig,axes=plt.subplots(len(REL),len(EVENTS),figsize=(7*len(EVENTS),3.2*len(REL)),sharex='col')
for i,(feat,label) in enumerate(REL):
    for j,ev in enumerate(EVENTS):
        ax=axes[i,j]; d=df[df.Event.eq(ev)&df.Clean]
        for comp in [c for c in COMP_ORDER if c in d.Compound.unique()]:
            g=d[d.Compound.eq(comp)]
            ax.scatter(g.TyreLife,g[feat],s=9,alpha=.18,color=PAL[comp],linewidths=0)
            b=g.groupby('TyreLife')[feat].agg(['median','size']); b=b[b['size']>=5]
            ax.plot(b.index,b['median'],color=PAL[comp],lw=2,label=comp)
        lo,hi=d[feat].quantile([.02,.98]); ax.set_ylim(lo-(hi-lo)*.05,hi+(hi-lo)*.05)
        ax.set_ylabel(label if j==0 else '')
        if i==0: ax.set_title(ev)
        if i==len(REL)-1: ax.set_xlabel('TyreLife [okr.]')
        if i==0: ax.legend(title='mieszanka',frameon=False)
plt.suptitle('Jak parametry zmieniają się z wiekiem opony',y=1.0,fontsize=14)
plt.tight_layout(); plt.savefig(REPORTS/'params_vs_tyre_life.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
# Macierz rozrzutu dla każdego toru (kolor = mieszanka)
PAIR=['TyreLife','DegDelta','CornerMinSpeed_mean','LatAcc_absmean','SpeedST','TrackTemp']
for ev in EVENTS:
    d=df[df.Event.eq(ev)&df.Clean][PAIR+['Compound']].dropna()
    d=d[d.DegDelta.between(*d.DegDelta.quantile([.01,.99]))]
    order=[c for c in COMP_ORDER if c in d.Compound.unique()]
    pg=sns.pairplot(d,vars=PAIR,hue='Compound',hue_order=order,palette=PAL,corner=True,height=2.1,
                    plot_kws={'s':8,'alpha':.3,'linewidth':0},diag_kws={'common_norm':False})
    pg.figure.suptitle(f'{ev}: macierz rozrzutu kluczowych parametrów',y=1.01,fontsize=14)
    pg.savefig(REPORTS/f'{slug(ev)}_pairplot.png',dpi=130); plt.show()

In [ ]:
# Zgodność torów: czy efekt wieku opony na cechę ma ten sam kierunek w Austrii i w Baku?
cmp=wear.pivot_table(index=['feature','group'],columns='Event',values='t_TyreLife').reset_index().dropna()
a,b=EVENTS
fig,ax=plt.subplots(figsize=(8.5,7.5))
GROUP_MARK={'telemetria':'o','pułapki prędkości':'s','kontekst':'^','cel':'D'}
for grp,g in cmp.groupby('group'):
    car=grp in ('telemetria','pułapki prędkości')
    ax.scatter(g[a],g[b],s=60,marker=GROUP_MARK.get(grp,'o'),color='#2a78d6' if car else '#9a9a9a',
               edgecolor='white',linewidth=1,label=grp,zorder=3)
for _,r in cmp.iterrows():
    if max(abs(r[a]),abs(r[b]))>=5 and r.feature!='LapTime': ax.annotate(r.feature,(r[a],r[b]),fontsize=8,xytext=(4,3),textcoords='offset points',color='#333')
lim=np.nanmax(np.abs(cmp[[a,b]].values))*1.1
ax.axhline(0,color='#666',lw=.8); ax.axvline(0,color='#666',lw=.8)
ax.fill_between([0,lim],0,lim,color='#2a78d6',alpha=.05); ax.fill_between([-lim,0],-lim,0,color='#2a78d6',alpha=.05)
ax.set(xlim=(-lim,lim),ylim=(-lim,lim),xlabel=f't(TyreLife) — {a}',ylabel=f't(TyreLife) — {b}',
       title='Zgodność torów: efekt wieku opony na parametr\n(zacienione ćwiartki = ten sam kierunek na obu torach)')
ax.legend(frameon=False,loc='lower right'); plt.tight_layout()
plt.savefig(REPORTS/'track_consistency_tyre_effect.png',dpi=150); plt.show()

## 8. Decyzja o pit stopie — kiedy zespoły zjeżdżały?
Etykieta pod przyszły model: `PitNextLap` = kierowca zjedzie na koniec następnego okrążenia. Cechy biorą wyłącznie dane znane **na koniec bieżącego okrążenia** (bez wycieku informacji z in-lapa).
Sprawdzamy:
- wiek opony w momencie zjazdu i wpływ SC/VSC,
- czy strata tempa (`DegDelta`) narasta przed zjazdem,
- które parametry niosą najwięcej informacji o `PitNextLap` (mutual information + korelacja punktowo-dwuseryjna).

In [ ]:
pits=df[df.IsPitInLap].copy()
pits['powód']=np.where(pits.UnderSC_VSC,'SC/VSC','zielona flaga')
display(pits.groupby(['Event','Compound','powód']).agg(n=('TyreLife','size'),tyre_life_median=('TyreLife','median'),
        tyre_life_min=('TyreLife','min'),tyre_life_max=('TyreLife','max'),lap_median=('LapNumber','median')))

fig,axes=plt.subplots(1,len(EVENTS),figsize=(7*len(EVENTS),4),sharey=True)
for ax,ev in zip(np.atleast_1d(axes),EVENTS):
    p=pits[pits.Event.eq(ev)]
    sns.histplot(data=p,x='LapNumber',hue='powód',multiple='stack',binwidth=1,ax=ax)
    sc=df[df.Event.eq(ev)&df.UnderSC_VSC].LapNumber.unique()
    for l in sc: ax.axvspan(l-.5,l+.5,color='orange',alpha=.12)
    ax.set_title(f'{ev}: okrążenie zjazdu (pomarańczowe tło = SC/VSC)')
plt.tight_layout(); plt.savefig(REPORTS/'pit_laps.png',dpi=160); plt.show()

# Strata tempa w ostatnich okrążeniach przed zjazdem (wyrównanie do in-lapa)
rows=[]
for (ev,drv,st),g in df.groupby(['Event','Driver','Stint']):
    g=g.sort_values('LapNumber')
    if not g.IsPitInLap.iloc[-1]: continue
    g=g.assign(laps_to_pit=g.LapNumber-g.LapNumber.iloc[-1])
    rows.append(g[g.Clean&g.laps_to_pit.ge(-15)][['Event','Compound','laps_to_pit','DegDelta']])
pre=pd.concat(rows)
plt.figure(figsize=(9,4))
sns.lineplot(data=pre,x='laps_to_pit',y='DegDelta',hue='Event',style='Compound',estimator='median',errorbar=('pi',50))
plt.axhline(0,color='k',lw=.6); plt.title('Strata tempa (po korekcie paliwa) przed zjazdem; 0 = in-lap')
plt.tight_layout(); plt.savefig(REPORTS/'degradation_before_pit.png',dpi=160); plt.show()

In [ ]:
from sklearn.feature_selection import mutual_info_classif
PIT_FEATURES=[c for c in FEATURES+['LapTimeNorm','DegDelta'] if c not in ('Stint',)]+['UnderSC_VSC']
mi_rows=[]
for ev in EVENTS+['OBIE RUNDY']:
    d=df if ev=='OBIE RUNDY' else df[df.Event.eq(ev)]
    d=d[~d.IsPitInLap & d.LapNumber.gt(1)]      # wiersz = koniec okrążenia, na którym jeszcze nie zjechał
    for feat in PIT_FEATURES:
        z=d[[feat,'PitNextLap']].dropna()
        if z[feat].nunique()<2 or z.PitNextLap.sum()<5: continue
        xv=z[feat].astype(float).to_numpy().reshape(-1,1)
        mi=mutual_info_classif(xv,z.PitNextLap,discrete_features=False,random_state=0)[0]
        mi_rows.append({'Event':ev,'feature':feat,'group':GROUP.get(feat,'cel/pochodna'),'MI':mi,
                        'r_pointbiserial':np.corrcoef(xv[:,0],z.PitNextLap)[0,1],'n':len(z),'n_pit':int(z.PitNextLap.sum())})
mi=pd.DataFrame(mi_rows)
mi_wide=mi.pivot_table(index=['feature','group'],columns='Event',values='MI').round(4)
mi_wide=mi_wide.sort_values('OBIE RUNDY',ascending=False).reset_index()
mi.to_csv(REPORTS/'pit_next_lap_information.csv',index=False)
display(mi_wide.head(20))
plt.figure(figsize=(9,6))
t=mi[mi.feature.isin(mi_wide.head(15).feature)]
sns.barplot(data=t,y='feature',x='MI',hue='Event',order=list(mi_wide.head(15).feature))
plt.title('Ile informacji o "zjedzie w następnym okrążeniu" niesie cecha (mutual information)')
plt.tight_layout(); plt.savefig(REPORTS/'pit_next_lap_mi.png',dpi=160); plt.show()

In [ ]:
# Parametry na okrążeniu przed zjazdem vs pozostałe okrążenia (bez okrążeń pit)
BOX=[('TyreLife','wiek opony [okr.]'),('DegDelta','strata vs start stintu [s]'),('LapTimeNorm','czas / mediana rundy'),('TrackTemp','temp. toru [°C]')]
d=df[~df.IsPitInLap & df.LapNumber.gt(1)].copy()
d['Zjazd w nast. okr.']=np.where(d.PitNextLap,'tak','nie')
fig,axes=plt.subplots(1,len(BOX),figsize=(4.6*len(BOX),4.5))
for ax,(feat,label) in zip(axes,BOX):
    z=d[d[feat].notna()]
    if feat in ('DegDelta','LapTimeNorm'): z=z[z[feat].between(*z[feat].quantile([.01,.99]))]
    sns.boxplot(data=z,x='Event',y=feat,hue='Zjazd w nast. okr.',hue_order=['nie','tak'],palette={'nie':'#c9c9c9','tak':'#eb6834'},
                ax=ax,showfliers=False,width=.6,linewidth=1)
    ax.set_xticks(range(len(EVENTS)),[e.split()[0] for e in EVENTS]); ax.set(xlabel='',ylabel=label)
    if ax is not axes[0]: ax.get_legend().remove()
plt.suptitle('Okrążenie przed pit stopem vs pozostałe okrążenia wyścigu',y=1.02,fontsize=14)
plt.tight_layout(); plt.savefig(REPORTS/'pit_vs_rest_boxplots.png',dpi=150,bbox_inches='tight'); plt.show()

## 9. Podsumowanie cech kandydackich
Jedna tabela łącząca wyniki: wpływ wieku opony (sekcja 7), informacja o pit stopie (sekcja 8) i nadmiarowość (sekcja 6). To punkt wyjścia do wyboru cech modelu.

In [ ]:
summary=wear_wide[['feature','group','same_sign_both','min_abs_t']+tcols].merge(
    mi_wide[['feature','OBIE RUNDY']].rename(columns={'OBIE RUNDY':'MI_pit_next_lap'}),on='feature',how='outer')
red_map=redundant.groupby('param_a').param_b.apply(lambda s:', '.join(sorted(set(s))))
red_map=pd.concat([red_map,redundant.groupby('param_b').param_a.apply(lambda s:', '.join(sorted(set(s))))]).groupby(level=0).agg(', '.join)
summary['redundant_with']=summary.feature.map(red_map)
summary['group']=summary.feature.map(GROUP).fillna(summary['group']).fillna('cel/pochodna')
# wskaźnik zużycia = tylko sygnały z auta (kontekst typu Position/TrackTemp zmienia się z czasem wyścigu, a nie z oponą)
summary['wear_indicator']=summary.same_sign_both.fillna(False)&summary.min_abs_t.ge(3)&summary.group.isin(['telemetria','pułapki prędkości'])
summary=summary.sort_values(['wear_indicator','min_abs_t'],ascending=False)
summary.to_csv(REPORTS/'feature_candidates_summary.csv',index=False)
display(summary.round(3))
df.to_parquet(CACHE/'analysis_dataset.parquet',index=False)   # zbiór gotowy pod modelowanie
print('Zapisano zbiór do modelowania:',CACHE/'analysis_dataset.parquet',df.shape)

## 10. Eksport raportów
Pliki są na Google Drive (`MyDrive/F1-Research/reports`). Dodatkowo pakujemy je do ZIP i pobieramy na komputer.

In [ ]:
zip_path=ROOT/'reports.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in REPORTS.glob('*'): z.write(p,p.name)
print('ZIP:',zip_path,f'{zip_path.stat().st_size/1024**2:.1f} MB')
if IN_COLAB and DOWNLOAD_REPORTS_ZIP: colab_files.download(str(zip_path))

## 11. Wnioski z pierwszego uruchomienia (2 tory, wyścigi 2025)
- **Degradacja jest wyraźna w Austrii, słaba w Baku.** Austria: ok. +0.05–0.06 s/okr. (MEDIUM i HARD), strata ~1–1.5 s po 25–30 okrążeniach. Baku: HARD ~0.015 s/okr., MEDIUM nieistotna statystycznie — tor uliczny o niskiej degradacji, gdzie silna ewolucja nawierzchni maskuje zużycie (dlatego `DegDelta` dla MEDIUM schodzi poniżej zera). Model musi znać „charakter toru”.
- **Najlepszy sygnał zużycia z telemetrii: prędkość w zakrętach** (`CornerMinSpeed_mean`, `Speed_mean`, `LatAcc_absmean`) — spada z wiekiem opony po kontroli paliwa i kierowcy, na obu torach w tym samym kierunku; w Austrii bardzo mocno (|t|≈20), w Baku słabiej (|t|≈4–5). Najmocniej w zakrętach średnich i wolnych.
- **Grupy nadmiarowe:** {`Speed_mean`, `CornerMinSpeed_mean`, `LatAcc_absmean`, `LatLoad`}, {`SpeedST`, `Speed_max`}, {`LapNumber`, `RaceProgress`, `TrackTemp`, `Pressure`} — do modelu po jednej cesze z grupy.
- **Pułapki:** `Position` i `TrackTemp` korelują z wiekiem opony przez cykl pit stopów i upływ czasu — to nie wskaźniki zużycia.
- **Decyzja o pit stopie:** pojedyncze cechy niosą mało informacji (MI ≤ 0.02); najwięcej `TyreLife`, prędkość w zakrętach i postęp wyścigu. Zjazd zależy od strategii (SC/VSC, okno pit stopu, rywale), więc model powinien łączyć zużycie z kontekstem wyścigu, a 2 wyścigi (~54 pit stopy) to mało przykładów — warto dodać kolejne rundy (wystarczy dopisać je do `EVENTS`, pobiorą się tylko nowe).